# Machine Learning Pipeline: Feature Importance Analysis

This notebook analyzes the `data.xlsx` dataset to find the most associated factors (features) that contribute to a specific target variable (e.g., Mental Exhaustion, Stress, or Academic Satisfaction).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Load the dataset
df = pd.read_excel('data.xlsx')
print("Dataset Shape:", df.shape)
df.head()

## Preprocessing
Most of the data is categorical (e.g., Likert scale responses). We will use `LabelEncoder` to convert these text values into numerical values so that the ML models can process them.

In [ ]:
# Handle missing values by filling them with a placeholder
df = df.fillna('Missing')

# Encode categorical variables
label_encoders = {}
df_encoded = df.copy()

for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object':
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le

df_encoded.head()

## Select Target Variable
We need to choose a target variable to predict. Let's use **'আমি আমার পড়াশোনার কারনে মানসিকভাবে ক্লান্ত বোধ করি'** (I feel mentally exhausted because of my studies) as an example target. 

*You can change `target_col` to any other column you want to analyze.*

In [ ]:
# Define the target column (Update this to your preferred target variable)
target_col = 'আমি আমার পড়াশোনার কারনে মানসিকভাবে ক্লান্ত বোধ করি'

if target_col not in df.columns:
    # Fallback if the column is not found (pick a valid column)
    target_col = df.columns[-5]

print(f"Target Variable: {target_col}")

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model 1: Random Forest Feature Importance
Random Forest provides a natural way to calculate the importance of each feature based on how much it decreases impurity across all trees in the forest.

In [ ]:
# Train a Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Get feature importances
importances = rf_model.feature_importances_
feature_names = X.columns

# Create a DataFrame for visualization
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top 10 Most Associated Factors (Random Forest):")
display(feature_importance_df.head(10))

## Model 2: Correlation Analysis
We can also look at the absolute Pearson correlation between the features and the target variable. This is a simpler linear approach.

In [ ]:
correlations = df_encoded.corr()[target_col].drop(target_col)
corr_df = pd.DataFrame({
    'Feature': correlations.index,
    'Correlation': correlations.values,
    'Abs_Correlation': np.abs(correlations.values)
}).sort_values(by='Abs_Correlation', ascending=False)

print("Top 10 Most Associated Factors (Correlation):")
display(corr_df.head(10))

## Visualization
Visualizing the top features from the Random Forest model.

In [ ]:
# Plot the top 10 features
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10))
plt.title(f'Top 10 Factors Associated with:\n"{target_col}"')
plt.xlabel('Feature Importance')
plt.ylabel('Factors')

# Note: Bengali fonts might appear as squares depending on OS/Jupyter config. 
# You may need to set a specific font family if that happens.
# plt.rcParams['font.family'] = 'Kalpurush' 

plt.tight_layout()
plt.show()